# 🧪 معمل التجارب: تحديث فهرس المتجهات وإضافة مستندات جديدة (Lecture 29)
دليل عملي لتحديث قاعدة بيانات المتجهات `ChromaDB` بإضافة مستندات وتضمينات جديدة فوراً عبر `add_documents` دون الحاجة لإعادة بناء الفهرس من الصفر، مع شرح آلية عمل كل خطوة.

### 1. استيراد المكتبات وتحديد مسار قاعدة البيانات المركزي
**آلية العمل:**
نقوم بتحميل متغيرات البيئة واستيراد مكونات LangChain الأساسية:
- كائن `Document` لتغليف النص الجديد مع البيانات الوصفية (`Metadata`).
- كائن `Chroma` و `HuggingFaceEmbeddings` للاتصال بالفهرس المخزن على القرص.
- `RecursiveCharacterTextSplitter` لتقطيع المستند الجديد قبل إضافته.
- `ChatGroq` وسلسلة `LCEL` لاختبار استرجاع وتوليد الإجابة من المستند الجديد.

In [13]:
# import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized database path
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
docs_dir = data_dir / "chroma_sample_data"
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# import langchain components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Libraries imported successfully!")
print(f"Database Directory: {db_dir}")

Libraries imported successfully!
Database Directory: ..\..\data\databases\chroma_vector_store


### 2. الاتصال بفهرس المتجهات الموجود مسبقاً وفحص عدد السجلات
**آلية العمل:**
نقوم بالاتصال بفهرس `ChromaDB` الموجود على القرص باستخدام نفس نموذج التضمين (`sentence-transformers/all-MiniLM-L6-v2`) واسم المجموعة (`rag_collection`).
ثم نفحص عدد المتجهات المخزنة حالياً باستخدام `_collection.count()` لمعرفة العدد قبل التحديث.

In [14]:
# initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect to the existing persistent chroma database
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

# get current document count before adding new data
initial_count = vector_store._collection.count()
print(f"Connected to vector store at: {db_dir}")
print(f"Document count BEFORE update: {initial_count}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to vector store at: ..\..\data\databases\chroma_vector_store
Document count BEFORE update: 16


### 3. إنشاء المستند الجديد وتزويده ببيانات وصفية (Metadata)
**آلية العمل:**
في سيناريوهات العمل الواقعية، تصل مستندات جديدة باستمرار إلى النظام.
هنا ننشئ كائن `Document` يحوي نصاً جديداً عن **الرؤية الحاسوبية (Computer Vision)** لم يكن موجوداً من قبل في الفهرس، ونضيف معه بيانات وصفية (`source: docs_3.txt`) لتتبع مصدر المعلومة.
كما نقوم بحفظ الملف في مجلد `chroma_sample_data` ليظل الأرشيف النصي متزامناً ومكتملاً.

In [15]:
# define content for the new document (Computer Vision)
cv_content = (
    "computer vision is a field of artificial intelligence that trains computers to interpret "
    "and understand the visual world. Using digital images from cameras and videos and deep learning models, "
    "machines can accurately identify and classify objects, and then react to what they see. "
    "Key applications include image classification, object detection, and visual recognition."
)

# create new Document instance with metadata
new_docs = Document(
    page_content=cv_content,
    metadata={"source": "docs_3.txt", "topic": "computer_vision"}
)

# save raw text file to data directory for consistency
new_doc_path = docs_dir / "docs_3.txt"
new_doc_path.write_text(new_docs.page_content, encoding="utf-8")

print(f"New document prepared with metadata: {new_docs.metadata}")
print(f"Saved text copy to: {new_doc_path.name}")

New document prepared with metadata: {'source': 'docs_3.txt', 'topic': 'computer_vision'}
Saved text copy to: docs_3.txt


### 4. تقطيع المستند الجديد إلى Chunks
**آلية العمل:**
قبل إضافة أي مستند لقاعدة المتجهات، يجب تمريره على الـ Text Splitter لتقسيمه إلى أجزاء بحجم مناسب للبحث الدلالي (`chunk_size=200`, `chunk_overlap=20`).
الميزة الأهم هي أن الـ Metadata تنتقل تلقائياً مع كل Chunk ناتج.

In [16]:
# initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

# split the new document into chunks
new_chunked_docs = text_splitter.split_documents([new_docs])

print(f"New document was split into {len(new_chunked_docs)} chunks:\n")
for idx, chunk in enumerate(new_chunked_docs, start=1):
    print(f"  Chunk [{idx}] ({len(chunk.page_content)} chars): {chunk.page_content[:70]}...")

New document was split into 2 chunks:

  Chunk [1] (192 chars): computer vision is a field of artificial intelligence that trains comp...
  Chunk [2] (193 chars): learning models, machines can accurately identify and classify objects...


### 5. إضافة القطع الجديدة للفهرس القائم (`add_documents`)
**آلية العمل:**
نستدعي دالة `vector_store.add_documents(new_chunked_docs, ids=ids)`.
ما يحدث في الكواليس:
1. يتم تحويل القطع الجديدة فقط إلى متجهات (Embeddings) بواسطة النموذج.
2. يتم إلحاق هذه المتجهات مباشرة بفهرس ChromaDB على القرص دون إعادة تضمين البيانات القديمة.
3. نمرر معرفات ثابتة (`ids`) مبنية على اسم الملف ورقم القطعة، مما يفعّل خاصية الـ Upsert لمنع تكرار البيانات نهائياً عند إعادة تشغيل الخلية.
4. نتحقق عبر `_collection.count()` من عدد المتجهات في الفهرس.

In [ ]:
# generate deterministic IDs to prevent duplicates upon re-running
ids = [f"{chunk.metadata.get('source', 'doc')}_chunk_{i}" for i, chunk in enumerate(new_chunked_docs)]

# add new chunks directly into the existing vector store using upsert
added_ids = vector_store.add_documents(new_chunked_docs, ids=ids)

# check updated count
updated_count = vector_store._collection.count()
print(f"Successfully processed {len(added_ids)} vectors with deterministic IDs!")
print(f"Assigned IDs: {added_ids}")
print(f"Count BEFORE: {initial_count}")
print(f"Count AFTER:  {updated_count}")

### 6. التحقق والاستعلام عبر مسار LCEL RAG الكامل
**آلية العمل:**
للتأكد من أن المعرفة الجديدة أصبحت جزءاً فعالاً من النظام:
1. ننشئ مسار RAG متكامل باستخدام LCEL ومشغل الربط (`|`).
2. نطرح سؤالاً عن **الموضوع الجديد (Computer Vision)**.
3. يقوم الـ `Retriever` بالبحث واستخراج القطع الجديدة التي أضفناها للتو.
4. يقوم نموذج اللغة (`Groq LLM`) بصياغة الإجابة المعتمدة حصراً على هذا السياق الجديد.

In [18]:
# helper function to format context chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# create retriever fetching top 3 matches
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# define context-grounded prompt template
template = """You are a helpful AI assistant. Answer the question using ONLY the provided context.
If you don't know the answer, say "I don't know."
Keep the answer concise and well-structured.

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# construct lcel rag pipeline
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# query the updated vector store
query = "What is computer vision and what does it train computers to do?"
answer = rag_chain.invoke(query)

print(f"❓ Question: {query}\n")
print(f"💬 Answer from Updated Store:\n{answer}\n")

print("=" * 60)
# verify retrieved chunks and metadata
retrieved_chunks = retriever.invoke(query)
print("📚 Retrieved Chunks Verification:")
for idx, doc in enumerate(retrieved_chunks, start=1):
    src = doc.metadata.get("source", "unknown")
    topic = doc.metadata.get("topic", "unknown")
    print(f"  [{idx}] (Source: {src} | Topic: {topic}): {doc.page_content[:70]}...")

❓ Question: What is computer vision and what does it train computers to do?

💬 Answer from Updated Store:
**Computer vision** is a field of artificial intelligence that trains computers to interpret and understand the visual world. It does this by using digital images from cameras and videos along with deep‑learning models.

📚 Retrieved Chunks Verification:
  [1] (Source: ..\..\data\chroma_sample_data\docs_3.txt | Topic: unknown): computer vision is a field of artificial intelligence that trains comp...
  [2] (Source: docs_3.txt | Topic: computer_vision): computer vision is a field of artificial intelligence that trains comp...
  [3] (Source: docs_3.txt | Topic: computer_vision): computer vision is a field of artificial intelligence that trains comp...


## 🎯 ملخص الدرس (المحاضرة 29):
1. **`add_documents`**: هي الطريقة القياسية والفعالة لتحديث قاعدة بيانات المتجهات وإضافة المعرفة الجديدة دون مسح أو إعادة تضمين المتجهات القديمة.
2. **البيانات الوصفية (`Metadata`)**: عنصر حيوي لتوثيق مصدر وتاريخ وموضوع كل معلومة، وتسمح بالفلترة المتقدمة (Metadata Filtering).
3. **التكامل الفوري**: المعرفة المضافة تصبح جاهزة فوراً للاسترجاع بواسطة الـ `Retriever` ومسارات `LCEL` دون أي تأخير أو إعادة تشغيل.